# 06 · Phase 5 — Snapshots and differences

**Input:** `webapp/db.sqlite3` (Phase 3) · **Output:** `data/snapshots/<label>/` (real baseline) and `docs/phase5_validation_report.json`

**Snapshot** = the database frozen at one moment: 9 CSV tables + `snapshot.json` (SHA-256 of each table, `content_id`, provenance) + `snapshot.jsonld` (the same as DCAT/PROV).
**Diff** = what changed between two snapshots: rows added, removed, changed (cell by cell), grouped by CVE.

We have **one real version** of the data (2025-07-28). To prove that the diff is *correct* we need a second version whose changes we know exactly. So:
1. freeze the **real** baseline;
2. make a **synthetic** second version with a list of changes written down *before* running the diff (ground truth);
3. check that the diff finds exactly those changes — and nothing else.

| Check family | Question |
|---|---|
| snapshot | does the snapshot of the database contain exactly the Phase-2 tables (same hashes)? |
| repro | same data ⇒ same `content_id`, on any run? append-only respected? |
| fixity | does `verify` catch every kind of damage? |
| diff | does the diff equal the ground truth, an independent recount, and `A + diff = B`? |
| trap | positional columns (`mapping_id`) and row order must NOT create fake changes |
| jsonld | is the snapshot description valid JSON-LD with the version link? |
| cli | do `python -m threatgraph.snapshot verify/diff` give the right exit codes? |
| negative | do the checks FAIL on broken data? |

In [1]:
import sys, os, io, json, shutil, subprocess, tempfile, copy
from pathlib import Path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "config" / "sources.yaml").exists())
WEBAPP = ROOT / "webapp"
sys.path[:0] = [str(ROOT / "src"), str(WEBAPP)]
os.environ["DJANGO_SETTINGS_MODULE"] = "threatgraph_site.settings"; os.environ["DJANGO_ALLOW_ASYNC_UNSAFE"] = "true"
import django; django.setup()
import numpy as np, pandas as pd
from IPython.display import display
from django.core.management import call_command
from rdflib import Graph, URIRef
from threatgraph import config, cleaning as C, jsonld as LD, snapshot as SN
from threatgraph.integrity import sha256_bytes, sha256_file
import catalog.models as M
pd.set_option("display.max_columns", 30, "display.max_colwidth", 80, "display.width", 200)
SRC = config.load_sources(); CLEAN = config.CLEAN_DIR; SNAPS = config.SNAPSHOT_DIR
clean_manifest = json.loads((CLEAN / "manifest.json").read_text())
assert json.loads((ROOT / "docs" / "phase4_validation_report.json").read_text())["all_passed"], "Phase 4 gate not passed"
assert M.Vulnerability.objects.count() == 1391, "database is empty: python manage.py load_clean_data"
TMP = Path(tempfile.mkdtemp(prefix="tg_snap_"))
checks = []
def check(family, name, ok, detail=""):
    checks.append({"family": family, "name": name, "ok": bool(ok), "detail": str(detail)[:300]}); return bool(ok)
LABEL = SRC["common_version"]["label"]; print("baseline label:", LABEL, "| temp dir:", TMP)

baseline label: 2025-07-28 | temp dir: /tmp/tg_snap_mz0fx60c


## 1 · The real baseline: freeze the database
`snapshot_db` reads the tables **back from the database** (not from `data/clean/`), serializes them with the Phase-2 function and writes the folder. Running it twice is harmless (same content ⇒ nothing happens).

In [2]:
out = io.StringIO(); call_command("snapshot_db", stdout=out); print(out.getvalue())
base = SNAPS / LABEL
man = SN.read_manifest(base)
display(pd.DataFrame(man["tables"]).T[["rows", "bytes", "sha256"]].assign(sha256=lambda d: d.sha256.str[:16] + "…"))
print("content_id :", man["content_id"]); print("provenance :", json.dumps(man["provenance"], indent=1)[:600])

snapshot /home/tavaa/Advanced_data_management/ThreatGraph/data/snapshots/2025-07-28
content_id 74d833caed5fc33635e2469d02e09b736a217ccc115460cbf7e203b521105e0a
  kev_vulnerability            1391 rows  fc0c9410db47…
  kev_cwe                      1328 rows  b84bc6036a20…
  nvd_cve                      1391 rows  435b99c03249…
  nvd_cve_cwe                  2523 rows  51ec143965c2…
  attack_tactic                  14 rows  7e2e881702e4…
  attack_technique              799 rows  912ea6914222…
  attack_technique_tactic      1051 rows  3008af112e5a…
  pkev_mapping                 1183 rows  6b76edf9253c…
  pkev_mapping_reference       2256 rows  c9111e106ef3…



,rows,bytes,sha256
kev_vulnerability,1391,706216,fc0c9410db476d57…
kev_cwe,1328,29558,b84bc6036a202b16…
nvd_cve,1391,724852,435b99c03249cd74…
nvd_cve_cwe,2523,148888,51ec143965c24292…
attack_tactic,14,1285,7e2e881702e4765d…
attack_technique,799,1195845,912ea691422248ae…
attack_technique_tactic,1051,16182,3008af112e5ab1b4…
pkev_mapping,1183,518104,6b76edf9253c3e84…
pkev_mapping_reference,2256,224856,c9111e106ef37a8d…


content_id : 74d833caed5fc33635e2469d02e09b736a217ccc115460cbf7e203b521105e0a
provenance : {
 "common_version": "2025-07-28",
 "raw_manifest_sha256": "f37bf97164198890f97e4135f5e1a128237e74b1e09afbc6c713db8a1f6e82e4",
 "cleaning_code_sha256": "2fb1c1f149e36af58a8937c246dff9555fa47c4c62633fedc6cea567e7f17cfa",
 "sources_digest_sha256": "33744c0d9ea01316aa73c5a425a241df0fd2010028b15377a6253c1196f0006b",
 "pinned_refs": {
  "kev": "0cca62ed469c333a45760beeaf35fcc53352aedd",
  "attack": "6cda5ad8462c79e14fbb872f4e09059b18e0cfc4",
  "pkev": "ddc842aa48c871cbfc2219cdbb842e9a4e155afe",
  "nvd": "a4262227a26a698199ea8d287932f8639e85aa6d"
 }
}


In [3]:
want = {t: clean_manifest["tables"][t]["sha256"] for t in SN.TABLES}
check("snapshot", "9 tables in the snapshot, SHA-256 of each == Phase-2 manifest (database export is lossless)", {t: i["sha256"] for t, i in man["tables"].items()} == want)
check("snapshot", "content_id == content_id computed from the Phase-2 manifest hashes (independent route)", man["content_id"] == SN.content_id(want), man["content_id"][:16])
check("snapshot", "row counts == Phase-2 manifest", {t: i["rows"] for t, i in man["tables"].items()} == {t: clean_manifest["tables"][t]["rows"] for t in SN.TABLES})
check("snapshot", "verify_snapshot finds no problem in the fresh snapshot", SN.verify_snapshot(base) == [], SN.verify_snapshot(base))
check("snapshot", "all files are read-only (append-only archive)", all(not os.access(p, os.W_OK) or getattr(os, 'geteuid', lambda: 1)() == 0 for p in base.rglob("*") if p.is_file()))
check("snapshot", "provenance records the raw-manifest and cleaning-code hashes of Phase 2",
      man["provenance"]["raw_manifest_sha256"] == clean_manifest["raw_manifest_sha256"] and man["provenance"]["cleaning_code_sha256"] == clean_manifest["cleaning_code_sha256"])

True

## 2 · Reproducibility and append-only
The `content_id` depends **only on the data bytes**. Not on the label, not on the creation time, not on the machine. So you can run this notebook on another computer and compare the id with the one printed above.

In [4]:
tabs = {t: C.read_clean(t, CLEAN) for t in SN.TABLES}
r1 = SN.create_snapshot(tabs, TMP / "r1", "run-1"); r2 = SN.create_snapshot(tabs, TMP / "r2", "other-label-and-time")
m1, m2 = SN.read_manifest(r1), SN.read_manifest(r2)
check("repro", "same data, different label/time/folder: identical table hashes and identical content_id (also == the real baseline)",
      m1["content_id"] == m2["content_id"] == man["content_id"] and m1["tables"] == m2["tables"], m1["content_id"][:16])
check("repro", "the files themselves are byte-identical", all((r1 / m1["tables"][t]["file"]).read_bytes() == (r2 / m2["tables"][t]["file"]).read_bytes() for t in SN.TABLES))
check("repro", "creating the same label again with the same data is a no-op (idempotent)", SN.create_snapshot(tabs, SNAPS, LABEL) == base)
broken = {**tabs, "attack_tactic": tabs["attack_tactic"].assign(name=lambda d: d.name.where(d.index != 0, "CHANGED"))}
try: SN.create_snapshot(broken, SNAPS, LABEL); ok = False
except SN.SnapshotExists as e: ok = True; msg = str(e)[:90]
check("repro", "same label, DIFFERENT data -> SnapshotExists (append-only: history cannot be rewritten)", ok, msg if ok else "")
check("repro", "no half-written folder is left behind (.part)", not list(SNAPS.glob("*.part")) and not list(TMP.glob("*/*.part")))

True

## 3 · Fixity: can `verify` catch damage?
Copy the snapshot, damage the copy in six different ways, expect a different complaint each time. (A check that cannot fail proves nothing — these are the negative tests of the fixity family.)

In [5]:
def fresh_copy(name):
    d = TMP / name; shutil.copytree(base, d); SN.make_writable(d); return d
neg = []
def negative(name, detected, detail=""):
    neg.append({"damage / attack": name, "detected": bool(detected), "what verify says": str(detail)[:110]}); check("negative", name, detected, detail)
d = fresh_copy("t1"); f = d / "tables" / "kev_vulnerability.csv"; b = bytearray(f.read_bytes()); b[500] ^= 1; f.write_bytes(bytes(b))
p = SN.verify_snapshot(d); negative("one bit flipped inside a table file", any("SHA-256 differs" in x for x in p), p)
d = fresh_copy("t2"); (d / "tables" / "nvd_cve.csv").unlink(); p = SN.verify_snapshot(d); negative("a table file deleted", any("missing" in x for x in p), p)
d = fresh_copy("t3"); (d / "tables" / "notes.txt").write_text("hello"); p = SN.verify_snapshot(d); negative("an unexpected file added", any("unexpected" in x for x in p), p)
d = fresh_copy("t4"); mj = json.loads((d / "snapshot.json").read_text()); mj["tables"]["kev_cwe"]["sha256"] = "0" * 64; (d / "snapshot.json").write_text(json.dumps(mj))
p = SN.verify_snapshot(d); negative("manifest edited (a table hash replaced)", any("kev_cwe" in x for x in p) and any("content_id" in x for x in p), p)
d = fresh_copy("t5"); f = d / "tables" / "pkev_mapping.csv"; f.write_bytes(f.read_bytes()[:-200]); p = SN.verify_snapshot(d); negative("table file truncated", any("size" in x for x in p), p)
d = fresh_copy("t6"); (d / "snapshot.json").write_text("{not json"); p = SN.verify_snapshot(d); negative("manifest destroyed", any("unreadable" in x for x in p), p)
display(pd.DataFrame(neg))
check("fixity", f"{sum(n['detected'] for n in neg)}/{len(neg)} kinds of damage detected by verify_snapshot", all(n["detected"] for n in neg))

,damage / attack,detected,what verify says
0,one bit flipped inside a table file,True,['kev_vulnerability: SHA-256 differs from manifest']
1,a table file deleted,True,['nvd_cve: file missing']
2,an unexpected file added,True,['unexpected file tables/notes.txt']
3,manifest edited (a table hash replaced),True,"['kev_cwe: SHA-256 differs from manifest', 'content_id does not match the ta..."
4,table file truncated,True,"['pkev_mapping: size 517904 != manifest 518104', 'pkev_mapping: SHA-256 diff..."
5,manifest destroyed,True,['manifest unreadable: Expecting property name enclosed in double quotes: li...


True

## 4 · The diff
### 4.1 Diff of identical snapshots, and order independence

In [6]:
A = SN.read_tables(base)                                   # frames of STRINGS, exactly what is on disk ('' = NULL)
d0 = SN.diff_snapshots(base, r1)
check("diff", "diff(baseline, its byte-identical copy) is empty in all 9 tables", d0.identical and d0.summary().to_numpy().sum() == 0)
shuf = {t: df.sample(frac=1, random_state=3).reset_index(drop=True) for t, df in A.items()}
check("trap", "row order does not matter: diff(A, A with every table shuffled) is empty", SN.diff_tables(A, shuf).identical)

True

### 4.2 The synthetic second version, with its ground truth
Every operation below is written **with its expected effect** (`exp`), counted by hand while the change is made — not by the diff code.
The most important operation is the last one: a new mapping is inserted at the **top** of the PKEV file, so the position number `mapping_id` of *every* other mapping goes up by one. A naive diff would report ~1,000 changes; the right answer is only the real ones.

In [7]:
B = {t: df.copy() for t, df in A.items()}
exp = {t: dict(added=0, removed=0, changed_rows=0, changed_cells=0) for t in SN.TABLES}      # ground truth, kept by hand
touched, log = set(), []
def note(op, text): log.append((op, text))
pm, pr, kv = A["pkev_mapping"], A["pkev_mapping_reference"], A["kev_vulnerability"]

def drop(t, mask):
    n = int(mask.sum()); B[t] = B[t][~mask]; exp[t]["removed"] += n; return n
def change(t, mask, col, new):
    assert mask.sum() == 1; B[t].loc[mask, col] = new; exp[t]["changed_cells"] += 1
def row_changed(t): exp[t]["changed_rows"] += 1

# OP2 - remove a whole CVE (KEV + NVD + CWE + mappings + evidence)
cnt = pm.cve_id.value_counts(); multi = sorted(cnt[cnt >= 2].index)
cve_rm, cve_maprm, cve_cmt = multi[0], multi[1], multi[2]; touched |= {cve_rm, cve_maprm, cve_cmt}
mids = set(pm.loc[pm.cve_id == cve_rm, "mapping_id"])
n = {t: drop(t, B[t].cve_id == cve_rm) for t in ["kev_vulnerability", "kev_cwe", "nvd_cve", "nvd_cve_cwe", "pkev_mapping"]}
n["pkev_mapping_reference"] = drop("pkev_mapping_reference", B["pkev_mapping_reference"].mapping_id.isin(mids))
note("OP2", f"{cve_rm} removed everywhere: " + ", ".join(f"{k} -{v}" for k, v in n.items() if v))

# OP7 - remove ONE mapping (and its evidence) of another CVE
mid = pm.loc[pm.cve_id == cve_maprm].mapping_id.iloc[0]; tq = pm.loc[pm.mapping_id == mid].technique_id.iloc[0]
drop("pkev_mapping", B["pkev_mapping"].mapping_id == mid); k = drop("pkev_mapping_reference", B["pkev_mapping_reference"].mapping_id == mid)
note("OP7", f"{cve_maprm}: mapping to {tq} removed (+{k} evidence rows)")

# OP8 - change the comment of one mapping
row = pm[(pm.cve_id == cve_cmt) & (pm.comments != "")].iloc[0]
change("pkev_mapping", B["pkev_mapping"].mapping_id == row.mapping_id, "comments", row.comments + " [reviewed]"); row_changed("pkev_mapping")
note("OP8", f"{cve_cmt}: comment of mapping to {row.technique_id} extended")

# OP3 - ransomware flag
cve_rw = sorted(set(kv[kv.ransomware_use == "unknown"].cve_id) - touched)[0]; touched.add(cve_rw)
change("kev_vulnerability", B["kev_vulnerability"].cve_id == cve_rw, "ransomware_use", "known"); row_changed("kev_vulnerability")
note("OP3", f"{cve_rw}: ransomware_use unknown -> known")

# OP4 - NVD rescoring (3 cells, 1 row)
nv = A["nvd_cve"]; cve_sc = sorted(set(nv[(nv.cvss_v31_score == "9.8") & (nv.cvss_v31_severity == "CRITICAL")].cve_id) - touched)[0]; touched.add(cve_sc)
old_lm = nv.loc[nv.cve_id == cve_sc, "last_modified"].iloc[0]; m_ = B["nvd_cve"].cve_id == cve_sc
for col, new in [("cvss_v31_score", "7.5"), ("cvss_v31_severity", "HIGH"), ("last_modified", "2025-08-01" + old_lm[10:])]: change("nvd_cve", m_, col, new)
row_changed("nvd_cve"); note("OP4", f"{cve_sc}: CVSS 9.8 CRITICAL -> 7.5 HIGH (+ last_modified)")

# OP5 - NVD status
cve_st = sorted(set(nv[nv.vuln_status == "Analyzed"].cve_id) - touched)[0]; touched.add(cve_st)
old_lm = nv.loc[nv.cve_id == cve_st, "last_modified"].iloc[0]; m_ = B["nvd_cve"].cve_id == cve_st
change("nvd_cve", m_, "vuln_status", "Modified"); change("nvd_cve", m_, "last_modified", "2025-08-02" + old_lm[10:]); row_changed("nvd_cve")
note("OP5", f"{cve_st}: status Analyzed -> Modified (+ last_modified)")

# OP9 - ATT&CK side
at, tac = A["attack_technique"], A["attack_tactic"]
change("attack_technique", B["attack_technique"].technique_id == "T1003", "name", at.loc[at.technique_id == "T1003", "name"].iloc[0] + " (renamed)")
change("attack_technique", B["attack_technique"].technique_id == "T1003", "modified", "2025-08-03" + at.loc[at.technique_id == "T1003", "modified"].iloc[0][10:]); row_changed("attack_technique")
used = set(pm.technique_id); t_dep = sorted(set(at[at.status == "active"].technique_id) - used)[0]
change("attack_technique", B["attack_technique"].technique_id == t_dep, "status", "deprecated"); row_changed("attack_technique")
change("attack_tactic", B["attack_tactic"].tactic_id == "TA0001", "name", "Initial Access (v2)"); row_changed("attack_tactic")
tpl = at[at.technique_id == "T1003"].iloc[0].copy(); tpl["technique_id"] = "T9999"; tpl["stix_id"] = "attack-pattern--00000000-0000-0000-0000-000000009999"; tpl["name"] = "Synthetic technique"
tpl["is_subtechnique"] = "false"; tpl["parent_technique_id"] = ""
B["attack_technique"] = pd.concat([B["attack_technique"], tpl.to_frame().T], ignore_index=True); exp["attack_technique"]["added"] += 1
B["attack_technique_tactic"] = pd.concat([B["attack_technique_tactic"], pd.DataFrame([{"technique_id": "T9999", "tactic_id": "TA0001"}])], ignore_index=True); exp["attack_technique_tactic"]["added"] += 1
note("OP9", f"T1003 renamed (+modified), {t_dep} -> deprecated, TA0001 renamed, T9999 (+tactic link) added")

# OP6 - new mapping for a CVE that had none (+2 evidence rows); OP1 - brand-new CVE; mapping inserted at the TOP (shift)
cve_nm = sorted(set(kv.cve_id) - set(pm.cve_id) - touched)[0]; touched.add(cve_nm)
cve_new = "CVE-2099-0001"; touched.add(cve_new)
ROOT_T = sorted(set(kv.cve_id) - touched - (set(kv.cve_id) - set(A["nvd_cve_cwe"].cve_id)))[0]      # template CVE that has rows in KEV, NVD and NVD-CWE
def clone(t, new_id, **extra):
    r = A[t][A[t].cve_id == ROOT_T].iloc[:1].copy(); assert len(r) == 1, f"template {ROOT_T} has no row in {t}"   # (a template without a row would silently add nothing)
    r["cve_id"] = new_id
    for k, v in extra.items(): r[k] = v
    B[t] = pd.concat([B[t], r], ignore_index=True); exp[t]["added"] += 1
clone("kev_vulnerability", cve_new, src_row=str(int(kv.src_row.astype(int).max()) + 1)); clone("nvd_cve", cve_new); clone("nvd_cve_cwe", cve_new)
B["kev_cwe"] = pd.concat([B["kev_cwe"], pd.DataFrame([{"cve_id": cve_new, "cwe_id": "CWE-79"}])], ignore_index=True); exp["kev_cwe"]["added"] += 1
# shift of every existing position number, THEN the new rows
for t in ("pkev_mapping", "pkev_mapping_reference"): B[t]["mapping_id"] = (B[t]["mapping_id"].astype(int) + 1).astype(str)
mp_tpl = pm.iloc[0].copy()
def new_mapping(mid_, cve, tech, mtype):
    r = mp_tpl.copy(); r["mapping_id"] = str(mid_); r["cve_id"] = cve; r["technique_id"] = tech; r["mapping_type"] = mtype; r["comments"] = "synthetic"
    B["pkev_mapping"] = pd.concat([B["pkev_mapping"], r.to_frame().T], ignore_index=True); exp["pkev_mapping"]["added"] += 1
def new_ref(mid_, text, is_url):
    B["pkev_mapping_reference"] = pd.concat([B["pkev_mapping_reference"], pd.DataFrame([{"mapping_id": str(mid_), "reference": text, "is_http_url": is_url}])], ignore_index=True); exp["pkev_mapping_reference"]["added"] += 1
new_mapping(1, cve_new, "T1190", "exploitation_technique"); new_ref(1, "https://example.org/advisory", "true")
top = int(B["pkev_mapping"].mapping_id.astype(int).max()) + 1
new_mapping(top, cve_nm, "T1059.001", "primary_impact"); new_ref(top, "https://example.org/blog", "true"); new_ref(top, "private report, not a URL", "false")
note("OP6", f"{cve_nm}: NEW mapping to T1059.001 (+2 evidence rows)"); note("OP1", f"{cve_new}: brand-new CVE (KEV, CWE, NVD, NVD-CWE, mapping to T1190 at the TOP of the PKEV file, 1 evidence)")
note("SHIFT", "every other mapping_id +1 (position in the PKEV file)")
display(pd.DataFrame(log, columns=["operation", "what was done"]))
GT = pd.DataFrame(exp).T.rename(columns={"changed_rows": "changed rows", "changed_cells": "changed cells"}); display(GT)

,operation,what was done
0,OP2,"CVE-2009-3960 removed everywhere: kev_vulnerability -1, nvd_cve -1, nvd_cve_..."
1,OP7,CVE-2009-4324: mapping to T1204.002 removed (+1 evidence rows)
2,OP8,CVE-2010-0188: comment of mapping to T1105 extended
3,OP3,CVE-2002-0367: ransomware_use unknown -> known
4,OP4,CVE-2005-2773: CVSS 9.8 CRITICAL -> 7.5 HIGH (+ last_modified)
5,OP5,CVE-2010-5330: status Analyzed -> Modified (+ last_modified)
6,OP9,"T1003 renamed (+modified), T1001.001 -> deprecated, TA0001 renamed, T9999 (+..."
7,OP6,CVE-2004-0210: NEW mapping to T1059.001 (+2 evidence rows)
8,OP1,"CVE-2099-0001: brand-new CVE (KEV, CWE, NVD, NVD-CWE, mapping to T1190 at th..."
9,SHIFT,every other mapping_id +1 (position in the PKEV file)


,added,removed,changed rows,changed cells
kev_vulnerability,1,1,1,1
kev_cwe,1,0,0,0
nvd_cve,1,1,2,5
nvd_cve_cwe,1,1,0,0
attack_tactic,0,0,1,1
attack_technique,1,0,2,3
attack_technique_tactic,1,0,0,0
pkev_mapping,2,3,1,1
pkev_mapping_reference,3,3,0,0


In [8]:
snapB = SN.create_snapshot(None, TMP, "synthetic-B", SN.provenance_from(clean_manifest, SRC), previous=base,
                           csv_bytes={t: SN.frame_to_csv(B[t], t) for t in SN.TABLES})
check("snapshot", "synthetic snapshot B passes verify and has a different content_id from the baseline", SN.verify_snapshot(snapB) == [] and SN.read_manifest(snapB)["content_id"] != man["content_id"])
D = SN.diff_snapshots(base, snapB)                                      # read from disk, as a user would
S = D.summary(); display(S)
check("diff", "per-table counts (added / removed / changed rows / changed cells) == the ground truth written by hand", S.equals(GT.astype(S.dtypes.to_dict())), "")

,added,removed,changed rows,changed cells
table,,,,
kev_vulnerability,1,1,1,1
kev_cwe,1,0,0,0
nvd_cve,1,1,2,5
nvd_cve_cwe,1,1,0,0
attack_tactic,0,0,1,1
attack_technique,1,0,2,3
attack_technique_tactic,1,0,0,0
pkev_mapping,2,3,1,1
pkev_mapping_reference,3,3,0,0


True

### 4.3 What a person reads: the changelog by CVE

In [9]:
cl = D.by_cve()
for cve in [cve_new, cve_rm, cve_maprm, cve_cmt, cve_rw, cve_sc, cve_st, cve_nm]:
    print(cve); [print("   ", line) for line in cl[cve]]
print("\nnot tied to a CVE (ATT&CK tables):")
for t in ("attack_technique", "attack_technique_tactic", "attack_tactic"):
    d = D.tables[t]
    if len(d.added): print("  +", t, d.added.iloc[:, 0].tolist())
    for _, r in d.changed.iterrows(): print("  ~", t, r.iloc[0], f"[{r['column']}]", SN._short(r["old"]), "->", SN._short(r["new"]))
check("diff", "the CVEs in the changelog are exactly the 8 CVEs touched by the operations", set(cl) == touched, sorted(set(cl) ^ touched))
check("diff", "the changelog of the new CVE lists 6 additions; of the removed CVE only removals",
      sum(l.startswith("+") for l in cl[cve_new]) == 6 and all(l.startswith("-") for l in cl[cve_rm]))

CVE-2099-0001
    + kev_cwe CWE-79
    + kev_vulnerability
    + nvd_cve
    + nvd_cve_cwe NVD-CWE-noinfo/nvd@nist.gov
    + pkev_mapping T1190/exploitation_technique
    + pkev_mapping_reference T1190/exploitation_technique/https://example.org/advisory
CVE-2009-3960
    - kev_vulnerability
    - nvd_cve
    - nvd_cve_cwe NVD-CWE-noinfo/nvd@nist.gov
    - pkev_mapping T1190/exploitation_technique
    - pkev_mapping T1486/primary_impact
    - pkev_mapping_reference T1190/exploitation_technique/https://news.sophos.com/en-us…
    - pkev_mapping_reference T1486/primary_impact/https://news.sophos.com/en-us…
CVE-2009-4324
    - pkev_mapping T1204.002/exploitation_technique
    - pkev_mapping_reference T1204.002/exploitation_technique/https://trendmicro.com/vinfo/…
CVE-2010-0188
    ~ pkev_mapping T1105/primary_impact [comments] This vulnerability is exploited via dri… -> This vulnerability is exploited via dri…
CVE-2002-0367
    ~ kev_vulnerability [ransomware_use] unknown -> known
CVE-2005-

True

### 4.4 Is the diff *right*? Three independent proofs
1. **Independent recount** with plain Python sets (no pandas merge, different algorithm).
2. **Patch property:** baseline + diff must give exactly snapshot B (nothing missing, nothing extra).
3. **Inverse:** diff(B→A) must be diff(A→B) with added↔removed and old↔new swapped.

In [10]:
NA_, NB_ = SN.normalize(A), SN.normalize(SN.read_tables(snapB))
def recount(a, b):
    ra = {k: tuple(v) for k, v in zip(a.index, a.to_numpy().tolist())}; rb = {k: tuple(v) for k, v in zip(b.index, b.to_numpy().tolist())}
    common = set(ra) & set(rb)
    return dict(added=len(set(rb) - set(ra)), removed=len(set(ra) - set(rb)),
                changed_rows=sum(ra[k] != rb[k] for k in common), changed_cells=sum(x != y for k in common for x, y in zip(ra[k], rb[k])))
orc = pd.DataFrame({t: recount(NA_[t], NB_[t]) for t in SN.TABLES}).T.rename(columns={"changed_rows": "changed rows", "changed_cells": "changed cells"})
check("diff", "independent recount with Python sets gives the same table as the diff", orc.astype(S.dtypes.to_dict()).equals(S))
P = SN.apply_diff(A, D)
check("diff", "patch property: baseline + diff == snapshot B in all 9 tables (diff is complete and has nothing extra)", all(P[t].equals(NB_[t]) for t in SN.TABLES), [t for t in SN.TABLES if not P[t].equals(NB_[t])])
Dr = SN.diff_snapshots(snapB, base); Sr = Dr.summary()
check("diff", "inverse: diff(B->A) has added/removed swapped and the same number of changed cells", (Sr["added"] == S["removed"]).all() and (Sr["removed"] == S["added"]).all() and (Sr["changed cells"] == S["changed cells"]).all())
Pr = SN.apply_diff(SN.read_tables(snapB), Dr)
check("diff", "round trip: B + diff(B->A) == baseline", all(Pr[t].equals(NA_[t]) for t in SN.TABLES))

True

### 4.5 The traps: positions, and bytes vs meaning

In [11]:
m_a = A["pkev_mapping"].set_index(["cve_id", "technique_id", "mapping_type"]).mapping_id.astype(int)
m_b = SN.read_tables(snapB)["pkev_mapping"].set_index(["cve_id", "technique_id", "mapping_type"]).mapping_id.astype(int)
both = m_a.index.intersection(m_b.index); n_shift = int((m_a[both] != m_b[both]).sum())
print(f"mappings present in both versions: {len(both)} | of them with a DIFFERENT mapping_id in the file: {n_shift} | rows the diff calls 'changed' in pkev_mapping: {D.tables['pkev_mapping'].n_changed_rows}")
check("trap", "mapping_id shifted for (almost) every mapping, yet only the 1 really edited mapping is reported as changed", n_shift > 1000 and D.tables["pkev_mapping"].n_changed_rows == 1, (n_shift, D.tables["pkev_mapping"].n_changed_rows))
# bytes vs meaning: renumber ONLY the positions in a copy
sh = {t: df.copy() for t, df in A.items()}
for t in ("pkev_mapping", "pkev_mapping_reference"): sh[t]["mapping_id"] = (sh[t]["mapping_id"].astype(int) + 100).astype(str)
snapS = SN.create_snapshot(None, TMP, "positions-only", None, csv_bytes={t: SN.frame_to_csv(sh[t], t) for t in SN.TABLES})
check("trap", "positions renumbered only: content_id CHANGES (different bytes) but the diff is EMPTY (same meaning)",
      SN.read_manifest(snapS)["content_id"] != man["content_id"] and SN.diff_snapshots(base, snapS).identical)

mappings present in both versions: 1180 | of them with a DIFFERENT mapping_id in the file: 1180 | rows the diff calls 'changed' in pkev_mapping: 1


True

## 5 · The snapshot as JSON-LD (versions linked with PROV)

In [12]:
docA, docB = [json.loads((f / "snapshot.jsonld").read_text()) for f in (base, snapB)]
check("jsonld", "snapshot.jsonld of both snapshots uses only terms defined in the @context", not LD.undefined_terms({k: v for k, v in docA.items() if k != "@context"}) and not LD.undefined_terms({k: v for k, v in docB.items() if k != "@context"}))
g = Graph().parse(data=json.dumps(docA), format="json-ld").parse(data=json.dumps(docB), format="json-ld")
P_ = "PREFIX tg: <%s> PREFIX prov: <http://www.w3.org/ns/prov#> PREFIX dct: <http://purl.org/dc/terms/> " % LD.VOCAB
rev = [(str(a), str(b)) for a, b in g.query(P_ + "SELECT ?new ?old WHERE { ?new prov:wasRevisionOf ?old }")]
check("jsonld", "graph says: B prov:wasRevisionOf baseline (version lineage), by content-based IRIs", rev == [(LD.snapshot_iri(SN.read_manifest(snapB)["content_id"]), LD.snapshot_iri(man["content_id"]))], rev)
ids = {str(r[0]) for r in g.query(P_ + "SELECT ?c WHERE { ?s tg:contentId ?c }")}
check("jsonld", "contentId and 9 table nodes (rowCount, sha256) in the graph == snapshot.json", ids == {man["content_id"], SN.read_manifest(snapB)["content_id"]}
      and len(list(g.query(P_ + "SELECT ?t WHERE { ?s tg:tables ?t ; tg:contentId ?c }"))) == 18)
check("jsonld", "context still has one term per IRI after adding the snapshot terms", not LD.terms_sharing_an_iri())
print(json.dumps({k: v for k, v in docB.items() if k not in ("@context", "tables")}, indent=1))

{
 "@id": "urn:threatgraph:snapshot:78b836c469dced5b",
 "@type": "Dataset",
 "title": "ThreatGraph-LD snapshot synthetic-B",
 "version": "synthetic-B",
 "created": "2026-10-21T14:52:15Z",
 "contentId": "78b836c469dced5b87ed315e8b7dec40073d0480331566869909d73cbaa2a7c1",
 "wasRevisionOf": "urn:threatgraph:snapshot:74d833caed5fc336",
 "rawManifestSha256": "f37bf97164198890f97e4135f5e1a128237e74b1e09afbc6c713db8a1f6e82e4",
 "cleaningCodeSha256": "2fb1c1f149e36af58a8937c246dff9555fa47c4c62633fedc6cea567e7f17cfa"
}


## 6 · Command line (what you would run by hand)

In [13]:
env = {**os.environ, "PYTHONPATH": str(ROOT / "src")}
def cli(*args):
    r = subprocess.run([sys.executable, "-m", "threatgraph.snapshot", *args], capture_output=True, text=True, env=env, cwd=TMP); return r.returncode, r.stdout + r.stderr
rc, o = cli("verify", str(base)); print("$ verify baseline  ->", rc, "|", o.strip())
rc2, o2 = cli("diff", str(base), str(snapB)); print("$ diff baseline synthetic-B  ->", rc2); print(o2)
rc3, o3 = cli("diff", str(base), str(r1)); print("$ diff baseline copy  ->", rc3, "|", o3.strip().splitlines()[-1])
dmg = fresh_copy("t7"); (dmg / "tables" / "nvd_cve.csv").unlink()
rc4, o4 = cli("verify", str(dmg)); rc5, o5 = cli("diff", str(base), str(dmg))
check("cli", "exit codes: verify intact = 0, diff identical = 0, diff different = 1, verify damaged = 1, diff against damaged = 2",
      (rc, rc3, rc2, rc4, rc5) == (0, 0, 1, 1, 2), (rc, rc3, rc2, rc4, rc5))
check("cli", "the diff output names the changed tables", all(t in o2 for t in ("kev_vulnerability", "pkev_mapping", "attack_technique")))
rep_json = TMP / "diff.json"; cli("diff", str(base), str(snapB), "--json", str(rep_json)); dj = json.loads(rep_json.read_text())
check("cli", "--json writes the full diff (by_cve changelog included)", dj["identical"] is False and set(dj["by_cve"]) == touched)

$ verify baseline  -> 0 | OK: snapshot intact
$ diff baseline synthetic-B  -> 1
2025-07-28 (74d833caed5f) -> synthetic-B (78b836c469dc)
                         added  removed  changed rows  changed cells
table                                                               
kev_vulnerability            1        1             1              1
kev_cwe                      1        0             0              0
nvd_cve                      1        1             2              5
nvd_cve_cwe                  1        1             0              0
attack_tactic                0        0             1              1
attack_technique             1        0             2              3
attack_technique_tactic      1        0             0              0
pkev_mapping                 2        3             1              1
pkev_mapping_reference       3        3             0              0

$ diff baseline copy  -> 0 | IDENTICAL


True

## 7 · Negative tests of the *diff* — can it be fooled?

In [14]:
# (a) a diff that lost one addition must fail the patch property
Dbad = copy.deepcopy(D); Dbad.tables["kev_cwe"].added = Dbad.tables["kev_cwe"].added.iloc[0:0]
Pb = SN.apply_diff(A, Dbad); negative("diff with one lost addition -> patch property fails", not Pb["kev_cwe"].equals(NB_["kev_cwe"]))
# (b) wrong ground truth is rejected
GT2 = GT.copy(); GT2.loc["nvd_cve", "changed cells"] += 1; negative("ground truth with a wrong number -> comparison fails", not S.equals(GT2.astype(S.dtypes.to_dict())))
# (c) the independent recount sees a tampered snapshot
tam = {t: df.copy() for t, df in SN.read_tables(snapB).items()}; tam["kev_vulnerability"].loc[5, "product"] += "x"
tam_n = SN.normalize(tam); negative("recount on a tampered B differs from the diff", recount(NA_["kev_vulnerability"], tam_n["kev_vulnerability"]) != recount(NA_["kev_vulnerability"], NB_["kev_vulnerability"]))
# (d) broken input is refused, not silently accepted
bad = {t: df.copy() for t, df in A.items()}; bad["kev_cwe"] = pd.concat([bad["kev_cwe"], bad["kev_cwe"].iloc[:1]])
try: SN.normalize(bad); ok = False
except SN.DiffError as e: ok = True
negative("duplicate natural key in a table -> DiffError", ok)
bad = {t: df.copy() for t, df in A.items()}; bad["pkev_mapping_reference"].loc[0, "mapping_id"] = "99999"
try: SN.normalize(bad); ok = False
except SN.DiffError as e: ok = True
negative("evidence pointing to a mapping that does not exist -> DiffError", ok)
bad = {t: df.copy() for t, df in A.items()}; bad["attack_tactic"] = bad["attack_tactic"].drop(columns=["stix_id"])
try: SN.diff_tables(A, bad); ok = False
except SN.DiffError as e: ok = True
negative("tables with different columns -> DiffError", ok)
# (e) the diff refuses to compare a damaged snapshot
try: SN.diff_snapshots(base, dmg); ok = False
except SN.SnapshotError as e: ok = True
negative("diff against a damaged snapshot -> refused (never compare what you cannot trust)", ok)
display(pd.DataFrame(neg).tail(8))

,damage / attack,detected,what verify says
5,manifest destroyed,True,['manifest unreadable: Expecting property name enclosed in double quotes: li...
6,diff with one lost addition -> patch property fails,True,
7,ground truth with a wrong number -> comparison fails,True,
8,recount on a tampered B differs from the diff,True,
9,duplicate natural key in a table -> DiffError,True,
10,evidence pointing to a mapping that does not exist -> DiffError,True,
11,tables with different columns -> DiffError,True,
12,diff against a damaged snapshot -> refused (never compare what you cannot tr...,True,


## 8 · Phase gate

In [15]:
rep = pd.DataFrame(checks)
summary = rep.groupby("family").ok.agg(passed="sum", total="count"); display(summary)
failed = rep[~rep.ok]; print("FAILED:", failed.to_dict("records") if len(failed) else "none")
SN.make_writable(TMP); shutil.rmtree(TMP, ignore_errors=True)
assert failed.empty, "Phase 5 gate failed"
out = ROOT / "docs" / "phase5_validation_report.json"
out.write_text(json.dumps({"phase": 5, "all_passed": True, "baseline_label": LABEL, "baseline_content_id": man["content_id"],
                           "summary": summary.reset_index().to_dict("records"), "checks": checks,
                           "synthetic_ground_truth": GT.reset_index().rename(columns={"index": "table"}).to_dict("records")}, indent=2) + "\n")
print("PHASE 5 GATE: PASSED ->", len(checks), "checks | baseline content_id", man["content_id"][:16], "| report:", out.relative_to(ROOT))

,passed,total
family,,
cli,3,3
diff,8,8
fixity,1,1
jsonld,4,4
negative,13,13
repro,5,5
snapshot,7,7
trap,3,3


FAILED: none
PHASE 5 GATE: PASSED -> 44 checks | baseline content_id 74d833caed5fc336 | report: docs/phase5_validation_report.json


## 9 · What this does **not** prove
* The second version is **synthetic**: the diff is proved correct on changes we wrote down, not on a real upstream update. Real data will have surprises (e.g. a CVE changing vendor name, a technique being revoked and replaced).
* To get a **real** second version: change the pins in `config/sources.yaml`, re-run notebooks 01 → 04, then `python manage.py snapshot_db --label <new> --previous ../data/snapshots/<old>`; the same `diff` applies. This path is **not tested** [UNCERTAIN].
* `content_id` is a fixity hash, not a signature: whoever can rewrite a snapshot **and** its manifest can forge both. Protection against that would need an external, signed or timestamped record (not done).
* The diff works on the 9 tables only: it does not say *why* something changed, and a renamed natural key (e.g. a CVE id re-assigned) appears as one removal plus one addition.
* Snapshots duplicate the data (~3.5 MB each): fine here, not a storage strategy for large archives.

**Next (Phase 6, concept only):** source-coverage report, figures and tables for the report and the slides.